# Programmieraufgabe 6

Wie in den letzten beiden Aufgaben berechnen wir auch dieses mal wieder die Bahn einer Rakete zwischen Erde und Mond, jetzt mit einem Extrapolationsverfahren.

Tragen Sie zunächst in der folgenen Zelle Ihren Namen ein:

In [ ]:
# Numerik gewöhnlicher Differentialgleichungen
# Sommersemester 2026
# Übungsblatt 9 - Programmieraufgabe 6
#
# [Nachname], [Vorname]
# [Vorname.Nachname@uni-a.de]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from util.plotting_06 import plot_RK_solution
%matplotlib inline

Implementieren Sie zunächst das explizite Euler Verfahren und das Verfahren von Gragg (Seite 61 im Skript).

In [ ]:
def Euler(f, t0, u0, h, n_steps):
    '''
        Führt n_steps Schritte des expliziten Euler-Verfahrens durch
        Parameter:
            f:       rechte Seite der Differentialgleichung
            t0:      Start-Zeitpunkt
            u0:      Start-Funktionswert
            h:       Schrittweite
            n_steps: Anzahl Schritte
        Rückgabewert:
            u:       berechnete Approximation nach n_step Schritten
    '''
    u = u0.copy()
    ???
    return u

In [ ]:
def Gragg(f, t0, u0, h, n_steps):
    '''
        Führt n_steps Schritte des Verfahrens von Gragg durch
        Parameter:
            f:       rechte Seite der Differentialgleichung
            t0:      Start-Zeitpunkt
            u0:      Start-Funktionswert
            h:       Schrittweite
            n_steps: Anzahl Schritte
        Rückgabewert:
            u:       berechnete Approximation nach n_step Schritten
    '''
    ???
    return u

Sie können die nächste Zelle also formalen Test betrachten, ob Ihre Funktionen die richtigen Argumente akzeptieren und die korrekten Ausgaben liefern.

Beachten Sie wie immer, dass das nur eine Hilfestellung ist und ihre Funktionen auch Fehler enthalten können, die diese Tests nicht finden.

In [ ]:
f       = lambda t, y : np.array([t + y[0], t + y[1]])
t       = 2
y       = np.array([1.0, 2.0])
h       = 0.5
n_steps = 2

assert np.array_equal(Euler(f, t, y, h, n_steps), np.array([5.0, 7.25])), "Falscher Wert für das explizite Euler-Verfahren"
assert np.array_equal(Gragg(f, t, y, h, n_steps), np.array([6.5, 9.125])), "Falscher Wert für das Verfahren von Gragg"

Die folgenden beiden Funktionen dürfen Sie zur Bestimmung des Interpolationspolynoms verwenden. Beachten Sie dabei vorallem die Dimension des Rückgabewertes!

In [ ]:
def Lagrange_polynomial(j, nodes, points):
    '''
        Auswertung des j-ten Lagrange-Polynoms zu den Stützstellen nodes
        Parameter:
            j:        Grad des Polynoms
            nodes:    Stützstellen der Interpolation als np.array
            points:   Auswertungspunkte als np.array
        Rückgabewert:
            values:   Funktionswerte des Polynoms an den Auswertungspunkten als np.array
    '''
    values = np.ones_like(points)
    node_j = nodes[j]
    for node in nodes:
        if node != node_j:
            values *= (points - node) / (node_j - node)
    return values

def interpolation_polynomial(function_values, nodes, points):
    '''
        Auswertung des Interpolationspolynoms
        Parameter:
            fuction_values: Funktionswerte an den Stützstellen als np.array
            nodes:          Stützstellen der Interpolation als np.array
            points:         Auswertungspunkte als np.array
        Rückgabewert: 
            values:         Funktionswerte des Polynoms an den Auswertungspunkten als np.array
    '''
    values = np.zeros((function_values.shape[0], points.shape[0]))
    for i in range(function_values.shape[1]):
        values += np.outer(function_values[:,i], Lagrange_polynomial(i, nodes, points))
    return values

Implementieren Sie hier jetzt einen Makro-Schritt mit dem Extrapolationsverfahren (Seite 61 und 62). Da sowohl `Euler` als auch `Gragg` die selbe Signatur haben, kann die Berechnung einer Näherungslösung mit dem Basis-Verfahren und Schrittweite `h` in etwa folgendermaßen aussehen:
```
us_micro[:,i] = solver(f, t, y, h, n_micro[i])
```
Beachten Sie ausserdem den Parameter `k`: Für das Euler-Verfahren (`k = 1`) soll das Interpolationspolynom mit den Stützstellen $h_i$ berechnet werden, für Gragg (`k = 2`) aber mit $h_i^2$.

In [ ]:
def extrapolation_step(f, t, y, h_macro, n_micro, solver, k=2):
    '''
    Berechne einen Makro-Schritt des Extrapolationsverfahrens
    Parameter:
        f:         rechte Seite der Differentialgleichung
        t:         aktueller Zeitpunkt
        y:         aktueller Funktionswert
        h_macro:   Grundschrittweite H
        n_micro:   Folge der Mikro-Schrittzahlen pro Makro-Schritt
        solver:    Basis-Verfahren (gegeben als Name der Funktion, die einen Schritt berechnet)
        k:         h-Potenz in der lokalen Fehlerentwicklung des Basisverfahrens (1 für Euler, 2 für Gragg)
    Rückgabewert:
        u:         Approximation von y(t + h_macro)
    '''
    us_micro = np.zeros((y.shape[0], n_micro.shape[0]))

    ???
    
    return u

Wir verwenden wieder das mittlerweile bekannte System aus den vorherigen Aufgaben.

In [ ]:
def Erde_Mond_Rakete(t, y, mu):
    '''
        Rechte Seite der Differentialgleichung, die die Bahn 
        einer Raumsonde zwischen Erde und Mond beschreibt
        Parameter:
            t:     aktueller Zeitpunkt
            y:     aktueller Funktionswert (x_1, x'_1, x_2, x'_2)
            mu:    relative Masse des Mondes im Vergleich zur Masse Erde+Mond
        Rückgabewert:
            y_new: Wert der Funktion
    '''
    
    mu_hat = 1.0 - mu # relative Masse der Erde
    n1 =  ((y[0] + mu    )**2 + y[2] * y[2])**1.5
    n2 =  ((y[0] - mu_hat)**2 + y[2] * y[2])**1.5

    y_new = np.array([y[1], y[0], y[3], y[2]])
    y_new[1] += 2 * y[3] - mu_hat * (y[0] + mu) / n1 - mu * (y[0] - mu_hat) / n2
    y_new[3] -= 2 * y[1] + mu_hat *  y[2]       / n1 + mu *  y[2]           / n2
    return y_new

Experimentieren Sie jetzt mit verschiedenen Verfahren `method`, Schrittzahlen `n_steps` und Unterteilungsfolgen `n_micro`. Beschreiben Sie dann kurz Ihre Erkenntnisse. Da `u0` und `T` so gewählt sind, dass 2 komplette Orbits durchlaufen werden, können Sie den Abstand des Start- vom Endpunkt als Fehlermaß verwenden.

In [ ]:
mu = 0.012277470841006752  # relative Masse des Mondes
f = lambda t, u: Erde_Mond_Rakete(t, u, mu)

# Anfangswert
u0 = np.array([1.2, 0.0, 0.0, -1.04935750983035])
t0  = 0
T = 2 * 6.1917317137

methods = {"Euler": (Euler, 1), "Gragg": (Gragg, 2)}

method  = "Gragg"
n_steps = 1_000
n_micro = np.array([1, 2, 3, 4, 6, 8, 12, 16])


h = (T - t0) / n_steps
us = np.zeros((n_steps + 1, 4))
ts = np.linspace(t0, T, n_steps + 1)
us[0,:] = u0
for j, t in enumerate(ts[:-1]):
    us[j+1,:] = extrapolation_step(f, t, us[j,:], h, n_micro, methods[method][0], k=methods[method][1])

fig = plt.figure(figsize=(5, 5))
print(f'Fehler: {np.linalg.norm(us[-1,:] - u0)}')
plot_RK_solution(us, mu, method, xlims = (-1.5, 1.4), ylims=(-1.5, 1.5))

In [ ]:
# Beschreiben Sie hier kurz Ihre Erkenntnisse.
# Bei welcher Anzahl an Schrittweiten liefert Gragg korrekte Ergebnisse (mit einem Fehler kleiner als ca. 1e-2)? Bei welcher Euler? Verwenden Sie sowohl die Romberg als auch die Bulirsch-Folge bis zum Wert 16 und vergleichen Sie.
#
#